# **Data Visualisation and Communication - CA2**

Additional Variable Information about the dataset

**InvoiceNo**: Invoice number. Nominal, a 6-digit integral number uniquely assigned to each transaction. If this code starts with letter 'c', it indicates a cancellation.

**StockCode**: Product (item) code. Nominal, a 5-digit integral number uniquely assigned to each distinct product.

**Description**: Product (item) name. Nominal.

**Quantity**: The quantities of each product (item) per transaction. Numeric.

**InvoiceDate**: Invoice Date and time. Numeric, the day and time when each transaction was generated.

**UnitPrice**: Unit price. Numeric, Product price per unit in sterling.

**CustomerID**: Customer number. Nominal, a 5-digit integral number uniquely assigned to each customer.

**Country** Country name. Nominal, the name of the country where each customer resides. 


## **Index**

### **1. LIBRARIES**

In [21]:
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### **2. LOADING THE DATA**

In [22]:
## Changing the format file to a CSV
## Note: I had to replace the blank space between the words "Online" and "Retail" and add a "_" to make this code work.

FOLDER = r'C:\Users\felip\Documents\CA2-DataVisComms\DataVisComms-CA2' # This is the folder where the files will be.
XLSX_PATH = os.path.join(FOLDER, "Online_Retail.xlsx")
CSV_PATH = os.path.join(FOLDER, "Online_Retail.csv")

# Only convert if the CSV is not there yet
if not os.path.exists(CSV_PATH):
    print("( (!) CSV not found, converting the Excel file (this can take a few minutes)...")
    pd.read_excel(XLSX_PATH).to_csv(CSV_PATH, index=False)
    print("Done!")
else:
    print("======> CSV already exists, nothing to convert <======")

======> CSV already exists, nothing to convert <======


In [30]:
df = pd.read_csv(CSV_PATH, low_memory=False)

print("Rows:", df.shape[0], "| columns:", df.shape[1]) # printing how meny columns and rows.
df.head() # Checking the dataset

Rows: 541909 | columns: 8


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


### **3. HANDLING MISSING VALUES**

In [24]:
# Preparing the data types per columns.
df["InvoiceNo"]   = df["InvoiceNo"].astype(str).str.strip()
df["StockCode"]   = df["StockCode"].astype(str).str.strip()
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["CustomerID"]  = df["CustomerID"].astype("Int64").astype("string")
df["Description"] = df["Description"].str.strip()
df["Country"]     = df["Country"].str.strip()

raw_rows = len(df)
print("Raw rows:", raw_rows) # Total raw rows

Raw rows: 541909


In [27]:
# 1. Checking missing values
miss = df.isnull().sum()
print(pd.DataFrame({"missing": miss, "pct": (miss / len(df) * 100).round(2)}))

             missing    pct
InvoiceNo          0   0.00
StockCode          0   0.00
Description     1454   0.27
Quantity           0   0.00
InvoiceDate        0   0.00
UnitPrice          0   0.00
CustomerID    135080  24.93
Country            0   0.00


In [28]:
## Are the missing Descriptions non-sales?
no_desc = df[df["Description"].isnull()]
print("Missing description with price == 0:", (no_desc["UnitPrice"] == 0).mean())
print("Missing description with no customer:", no_desc["CustomerID"].isnull().mean())

Missing description with price == 0: 1.0
Missing description with no customer: 1.0


In [29]:
# 2. Handling missing values
desc_map = (df.dropna(subset=["Description"])
              .groupby("StockCode")["Description"]
              .agg(lambda s: s.mode().iloc[0]))
df["Description"] = df["Description"].fillna(df["StockCode"].map(desc_map))
print("Description still missing:", df["Description"].isnull().sum())   # expect 112

# CustomerID: do NOT impute, flag instead
df["is_guest"] = df["CustomerID"].isnull()
print("Guest rows:", df["is_guest"].sum())

Description still missing: 112
Guest rows: 135080


In [32]:
# 3. Handling missing values
# Filling from the StockCode if possible.
desc_map = (df.dropna(subset=["Description"])
              .groupby("StockCode")["Description"]
              .agg(lambda s: s.mode().iloc[0]))
df["Description"] = df["Description"].fillna(df["StockCode"].map(desc_map))
print("Description still missing:", df["Description"].isnull().sum())

# CustomerID: do NOT impute, flag instead
df["is_guest"] = df["CustomerID"].isnull()
print("Guest rows:", df["is_guest"].sum())

# The 112 rows that can't be filled have a price of 0.

Description still missing: 112
Guest rows: 135080


In [33]:
# 4. Flag invalid values.
# Based on the information provided by the author, is important to flag some columns to keep the data set
# as clean as possible.

is_cancel  = df["InvoiceNo"].str.upper().str.startswith("C")
neg_qty    = df["Quantity"] <= 0
bad_price  = df["UnitPrice"] <= 0
non_prod   = ~df["StockCode"].str.match(r"^\d{5}[A-Za-z]{0,2}$")
dup        = df.duplicated(keep="first")
unspec     = df["Country"].eq("Unspecified")

print("Cancelled (C):", is_cancel.sum())
print("Quantity <= 0:", neg_qty.sum(), "| < 0:", (df["Quantity"] < 0).sum(), "| == 0:", (df["Quantity"] == 0).sum())
print("UnitPrice <= 0:", bad_price.sum(), "| < 0:", (df["UnitPrice"] < 0).sum(), "| == 0:", (df["UnitPrice"] == 0).sum())
print("Non-product StockCodes:", non_prod.sum())
print("Exact duplicates:", dup.sum())
print("Country Unspecified:", unspec.sum())
print("C rows with positive qty (should be 0):", (is_cancel & (df['Quantity'] > 0)).sum())
print("Negative qty, not cancelled:", (neg_qty & ~is_cancel).sum())

# See which StockCodes were classed as non-products
print(df[non_prod].groupby("StockCode")["Description"]
        .agg(rows="size", example="first").sort_values("rows", ascending=False).head(20))


Cancelled (C): 9288
Quantity <= 0: 10624 | < 0: 10624 | == 0: 0
UnitPrice <= 0: 2517 | < 0: 2 | == 0: 2515
Non-product StockCodes: 2995
Exact duplicates: 5268
Country Unspecified: 446
C rows with positive qty (should be 0): 0
Negative qty, not cancelled: 1336
              rows                             example
StockCode                                             
POST          1256                             POSTAGE
DOT            710                      DOTCOM POSTAGE
M              571                              Manual
C2             144                            CARRIAGE
D               77                            Discount
S               63                             SAMPLES
BANK CHARGES    37                        Bank Charges
AMAZONFEE       34                          AMAZON FEE
CRUK            16                     CRUK Commission
DCGSSGIRL       13                     GIRLS PARTY BAG
DCGSSBOY        11                      BOYS PARTY BAG
gift_0001_20    10  Dotco

In [34]:
# 5. Handling invalid values. (using the last audit)
steps = [
    ("cancelled invoices (C)",   is_cancel),
    ("negative/zero quantity",   neg_qty),
    ("zero/negative price",      bad_price),
    ("non-product stock codes",  non_prod),
    ("exact duplicates",         dup),
]

remaining = pd.Series(True, index=df.index)
audit = [("raw rows", raw_rows)]
for name, mask in steps:
    hit = remaining & mask
    audit.append((f"removed: {name}", int(hit.sum())))
    remaining &= ~mask
audit.append(("clean rows", int(remaining.sum())))
print(pd.DataFrame(audit, columns=["step", "rows"]))

cancellations = df[is_cancel].copy()      # keep for return analysis
clean = df[remaining].copy()

clean["Revenue"] = clean["Quantity"] * clean["UnitPrice"]
clean["Country"] = clean["Country"].replace("Unspecified", "Unknown")


                               step    rows
0                          raw rows  541909
1   removed: cancelled invoices (C)    9288
2   removed: negative/zero quantity    1336
3      removed: zero/negative price    1181
4  removed: non-product stock codes    2379
5         removed: exact duplicates    5221
6                        clean rows  522504


In [35]:
# Checking the data again.

miss = clean.isnull().sum()
print(pd.DataFrame({"missing": miss, "pct": (miss / len(clean) * 100).round(2)}))
print("Rows:", len(clean))
print("Quantity min:", clean["Quantity"].min(), "| UnitPrice min:", clean["UnitPrice"].min())
print("Cancelled left:", clean["InvoiceNo"].str.upper().str.startswith("C").sum())   # expect 0
print("Duplicates left:", clean.duplicated().sum())                                    # expect 0

             missing    pct
InvoiceNo          0   0.00
StockCode          0   0.00
Description        0   0.00
Quantity           0   0.00
InvoiceDate        0   0.00
UnitPrice          0   0.00
CustomerID    131354  25.14
Country            0   0.00
is_guest           0   0.00
Revenue            0   0.00
Rows: 522504
Quantity min: 1 | UnitPrice min: 0.04
Cancelled left: 0
Duplicates left: 0


### **4. OUTLIERS**